# Chapter 39 — Troubleshooting, common anti-patterns, and debugging checklist

**Book role:** Professional practice  
**Suggested book space:** about 8 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Diagnose common Tkinter errors systematically.
- Recognize geometry-manager, callback, thread, and image-reference failures.
- Debug data/database mismatches using small fixtures and logs.
- Build a reproducible bug report.

## 1. Tkinter-specific failure modes

A blank window can mean the event loop never started, widgets were never managed by pack/grid/place, or an exception interrupted construction. A widget may not resize because its parent lacks row/column weights or because `sticky`/`expand` is missing. A disappearing image usually means the Python reference was not kept alive. “Application not responding” commonly points to blocking work in a callback.

When debugging, reduce the UI to a minimal reproduction and inspect the exact exception. Confirm the Tk version, Python executable, display availability, and widget parent. Do not add a second root window as a quick fix. Use logging or a status label for runtime context, but keep the full stack trace in a development log rather than flooding an end-user dialog.

### Worked example 1: Inspect widget and geometry state

The function assumes a real widget that belongs to a running Tk interpreter. Window dimensions can still be provisional before the event loop has processed geometry requests.

In [ ]:
def debug_widget(widget):
    """Call this from a running Tk application while debugging."""
    print("widget:", widget)
    print("class:", widget.winfo_class())
    print("mapped:", widget.winfo_ismapped())
    print("size:", widget.winfo_width(), widget.winfo_height())
    print("parent:", widget.master)
    print("manager:", widget.winfo_manager())

# Example use after constructing a widget: debug_widget(my_widget)

## 2. Data and database failure modes

A result that is consistently too large may be caused by a join that multiplies records or confusing order lines with orders. A zero revenue total can arise from an overly strict status filter, an incorrect dtype, all-missing numeric columns, or a date range boundary. In MongoDB, a filter can silently miss documents if the query uses a string date while the collection stores BSON Date values. Compare data types and query results on a small known fixture before blaming the chart.

Database “connection refused” differs from failed authentication, DNS failure, server selection timeout, or missing table/collection. Check one layer at a time: configuration is present, host/port are correct, server is running and reachable, credentials have permissions, database/collection exists, query parameters match the schema, and returned types align with the adapter contract.

### Worked example 2: Compare intermediate analytical outputs

Intermediate counts identify whether rows were lost during import, cleaning, or filtering. Add aggregate reconciliation checks for more complex pipelines.

In [ ]:
def explain_pipeline(raw, cleaned, filtered):
    print("Raw rows:", len(raw), "columns:", len(raw.columns))
    print("Cleaned rows:", len(cleaned), "columns:", len(cleaned.columns))
    print("Filtered rows:", len(filtered), "columns:", len(filtered.columns))
    print("Cleaned columns:", cleaned.columns.tolist())
    print("Filtered status counts:")
    if "status" in filtered:
        print(filtered["status"].value_counts(dropna=False))

# Call at each boundary when a dashboard total unexpectedly changes.

## 3. Reproducible debugging and bug reports

A useful bug report includes the exact steps, input file/schema (synthetic when possible), Python and package versions, operating system, expected result, actual result, error/log excerpt with secrets redacted, and whether the issue is repeatable. “It crashed” does not give enough information to reproduce the defect.

Use a sequence: reproduce with a tiny fixture, locate the boundary where actual data first diverges from expected data, add a test that fails for the defect, fix the logic, rerun the relevant suite, and test the original scenario. This prevents fixes that merely mask the error in the visible widget.

## Guided lab: Lab: troubleshooting playbook

Create a Markdown troubleshooting playbook with at least 15 symptoms and a disciplined diagnostic order. Include a blank window, frozen UI, missing image, blank Treeview, inaccurate KPI, empty filter result, malformed Excel, connection timeout, credentials failure, MongoDB date mismatch, and failed export. For each, provide a likely boundary to inspect and one minimal reproduction step.

## Review questions and practice

1. Why is a minimal reproduction valuable?
2. What commonly makes a Tkinter interface freeze?
3. How can a relational join inflate revenue?
4. Which facts should a reproducible bug report include?

## Further reading

- Tkinter docs: https://docs.python.org/3/library/tkinter.html
- Python traceback: https://docs.python.org/3/library/traceback.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.